# Sélection du taux d’apprentissage de CAMeLBERT

Nous comparons **1e-5, 2e-5 et 5e-5**, sans modifier les données ni les autres réglages. Le taux d’apprentissage règle l’amplitude des mises à jour des poids.

Le mode **full** lance les trois contrôles préalables puis **six adaptations sur toutes les données** : trois taux × deux tâches, avec la graine **42** seulement. Le meilleur taux est proposé séparément pour TSAC et MADAR sur le macro-F1 de validation. La confirmation ultérieure sur trois graines reste une autre phase.

**Prérequis :** dépôt à jour, environnement `environments/encoder` installé, corpus autorisés et poids épinglés présents. Sur Kaggle, le notebook appelle les scripts du dépôt cloné ; les sources privées doivent être raccordées avant l’exécution. Voir `docs/53_notebook_selection_encodeur.md`.

Les modes smoke, full et replay sont distincts. Le test officiel n’est pas évalué. Aucun modèle de l’application n’est remplacé.


## 1. Configuration

`full` est la valeur par défaut et lance des entraînements réels. `smoke` vérifie uniquement les petits essais et ne choisit aucun gagnant. `replay` lit et contrôle une campagne existante sans entraîner. `off` permet de vérifier les cellules sans charger de corpus ni de modèle.


In [ ]:
from pathlib import Path
import json
import os
import sys

explicit_root = os.environ.get('TUNISCOPE_PROJECT_ROOT')
candidates = ([Path(explicit_root)] if explicit_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/tuniscope'),
]
ROOT = next((p.resolve() for p in candidates
             if (p / 'scripts/run_encoder_screening.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir le notebook depuis le dépôt à jour ou définir TUNISCOPE_PROJECT_ROOT ; voir docs/53.')
sys.path.insert(0, str(ROOT / 'scripts'))
from encoder_screening_notebook_support import notebook_run, campaign_figures

TRAINING_MODE = os.environ.get('TUNISCOPE_SCREENING_MODE', 'full')
DEVICE = os.environ.get('TUNISCOPE_SCREENING_DEVICE', 'auto')
REPLAY_RUN = os.environ.get('TUNISCOPE_SCREENING_RUN',
    'outputs/encoder_finetuning/screening/2026-11-06_lr_screen_mps_smoke')
REPLAY_EXPECTED_MODE = os.environ.get('TUNISCOPE_SCREENING_REPLAY_MODE', 'full')
if TRAINING_MODE not in {'full', 'smoke', 'replay', 'off'}:
    raise ValueError('Choisir full, smoke, replay ou off')
plan = json.loads((ROOT / 'configs/camelbert_screening_v1.json').read_text())
print('Mode :', TRAINING_MODE, '| Appareil demandé :', DEVICE)
print('Taux comparés :', ', '.join(f'{x:g}' for x in plan['learning_rates']))
print('Criblage : graine 42. Confirmation future : graines 13, 42, 2026.')


## 2. Exécution et contrôle

Chaque candidat démarre depuis les mêmes poids généraux. Il passe d’abord un contrôle sur 100 exemples par partition et tâche. En full, les deux adaptations utilisent ensuite tout le train et toute la validation, sauvegardent le meilleur checkpoint et conservent les métriques par classe à chaque époque. Les modèles sont rechargés avant la synthèse.

Le mode replay contrôle les fichiers et les preuves de rejeu archivées ; il ne refait pas lui-même l’inférence des checkpoints. Une campagne incomplète ou un mode différent de celui demandé provoque une erreur visible.


In [ ]:
run_result = notebook_run(TRAINING_MODE, DEVICE, REPLAY_RUN, REPLAY_EXPECTED_MODE)
if run_result is not None:
    campaign_directory, campaign = run_result
    from IPython.display import display, Markdown
    display(Markdown((campaign_directory / 'summary.md').read_text()))
else:
    print('Mode off : aucun entraînement ni chargement de corpus.')


## 3. Courbes et matrices

Les figures proviennent des runs vérifiés : pertes train/validation, macro-F1 par époque et matrice de confusion du checkpoint retenu. Elles portent le taux, la graine et le périmètre exploratoire. En full, six PNG montrent les six adaptations. En smoke, douze PNG montrent les contrôles techniques ; leurs scores ne sélectionnent aucun taux.


In [ ]:
figure_paths = []
if run_result is not None:
    from IPython.display import Image, display
    figure_paths = campaign_figures(campaign_directory, campaign)
    for figure in figure_paths:
        display(Image(filename=str(figure)))
    print(f'{len(figure_paths)} images conservées dans les sous-dossiers figures/ des runs.')


## 4. Résultat de cette étape

Une campagne full propose un taux par tâche. Il reste à confirmer ces configurations avec les trois graines, puis à compléter les cartes, analyses et évaluations finales. Un score de validation élevé n’établit pas un gain sur le test. Les livrables L05, L07 et L12 restent partiels.


In [ ]:
if run_result is not None:
    print('Dossier de la campagne :', campaign_directory)
    print('Synthèse :', campaign_directory / 'summary.md')
    print('Métriques et empreintes :', campaign_directory / 'screening.json')
    print('Contrôles :', campaign_directory / 'campaign_verification.json')
    if campaign['selected_learning_rates'] is None:
        print('Contrôle technique uniquement : aucun taux sélectionné.')
    else:
        for corpus, rate in campaign['selected_learning_rates'].items():
            print(f'{corpus.upper()} : taux proposé {rate:g}, à confirmer sur trois graines.')
